# CountBoundedSlices

[Codility task](https://app.codility.com/programmers/trainings/7/count_bounded_slices/) - Exercise 7, Data Structures - *medium*

A slice `(P, Q)` with `0 <= P <= Q < N` is **bounded** if `max(A[P..Q]) - min(A[P..Q]) <= K`.

`solution(K, A)` returns the number of bounded slices, capped at `1,000,000,000`.

Example: `K = 2`, `A = [3, 5, 7, 6, 3]` -> `9`.

Assumptions: `N` in `[1..100,000]`, `K` in `[0..10^9]`, `A[i]` in `[-10^9..10^9]`.

## Idea: sliding window with monotonic deques

- Any sub-slice of a bounded slice is bounded. So for each right end `Q` the valid left ends form a range `[P, Q]`, and `P` never moves left as `Q` grows.
- For every `Q`, shrink `P` while `max - min > K`, then add `Q - P + 1` slices (all the slices ending at `Q`).
- The window max and min are kept in two deques of **indices**:
  - `max_q` holds decreasing values (front = maximum);
  - `min_q` holds increasing values (front = minimum);
  - when `P` moves past the index at a front, pop it.

Each index enters and leaves each deque at most once: O(N) time, O(N) space.

In [ ]:
from collections import deque

LIMIT = 1_000_000_000


def solution(K, A):
    max_q, min_q = deque(), deque()
    left = 0
    total = 0
    for right, value in enumerate(A):
        while max_q and A[max_q[-1]] <= value:
            max_q.pop()
        max_q.append(right)
        while min_q and A[min_q[-1]] >= value:
            min_q.pop()
        min_q.append(right)

        while A[max_q[0]] - A[min_q[0]] > K:
            left += 1
            if max_q[0] < left:
                max_q.popleft()
            if min_q[0] < left:
                min_q.popleft()

        total += right - left + 1
        if total > LIMIT:
            return LIMIT
    return total

## Official example

In [ ]:
assert solution(2, [3, 5, 7, 6, 3]) == 9
print("example OK")

## Edge cases

In [ ]:
assert solution(0, [5]) == 1                          # a single element
assert solution(0, [1, 1, 1]) == 6                    # K = 0, all equal: 3 + 2 + 1
assert solution(0, [1, 2, 3]) == 3                    # K = 0, all different: only single elements
assert solution(10**9, [-10**9, 10**9]) == 2          # difference 2*10^9 > K: the pair is NOT bounded
assert solution(10**9, [0, 10**9]) == 3               # difference exactly K is allowed
print("edge cases OK")

## Boundaries and the cap

In [ ]:
assert solution(5, [1, 2, 3, 4]) == 10                # every slice is bounded: N(N+1)/2

# the cap: N = 100000 equal elements -> 5,000,050,000 slices, capped
assert solution(0, [7] * 100_000) == LIMIT
print("boundaries OK")

## Brute-force check

Reference in O(N^2): extend each slice one element at a time, tracking its running min and max.

In [ ]:
import random


def brute(K, A):
    count = 0
    for p in range(len(A)):
        lo = hi = A[p]
        for q in range(p, len(A)):
            lo = min(lo, A[q])
            hi = max(hi, A[q])
            if hi - lo <= K:
                count += 1
            else:
                break                       # extending only widens the range
    return min(count, LIMIT)


rng = random.Random(3)
for _ in range(3000):
    n = rng.randint(1, 25)
    span = rng.choice([3, 10, 100])
    A = [rng.randint(-span, span) for _ in range(n)]
    K = rng.randint(0, 2 * span)
    assert solution(K, A) == brute(K, A), (K, A)
print("matches brute force on 3000 random cases")

## Performance

In [ ]:
import time

rng = random.Random(5)
N = 100_000
cases = {
    "random": (10**6, [rng.randint(-10**9, 10**9) for _ in range(N)]),
    "sorted ascending": (10**5, list(range(N))),
    "sorted descending": (10**5, list(range(N, 0, -1))),
    "all equal": (0, [42] * N),
    "zig-zag": (3, [i % 2 * 10**9 for i in range(N)]),
}
for name, (K, A) in cases.items():
    start = time.perf_counter()
    result = solution(K, A)
    print(f"{name:18s} -> {result:>13,} in {time.perf_counter() - start:.3f}s")

## Notes

- Why shrinking from the left is correct: boundedness is monotone, so once `(P, Q)` is too wide, every `(P', Q')` with `P' <= P` and `Q' >= Q` is too.
- Deques store **indices**, not values, so expired elements can be detected with `index < left`.
- Equal values: `<=` / `>=` while popping keeps one representative; either choice works as long as the deque stays monotone.
- Python integers never overflow; the cap is applied only because the task asks for it.